<!-- 학습 보강 셀 -->

# 01. Basic Pipeline 학습 흐름

이 노트북은 RAG의 최소 실행 단위를 한 번에 연결해 보는 예제입니다.
흐름은 `문서 로드 -> 임베딩 -> VectorStoreIndex 생성 -> QueryEngine 질의 -> 근거 확인` 순서입니다.
각 셀을 실행할 때 지금 다루는 객체가 `Document`, `Index`, `QueryEngine`, `Response` 중 무엇인지 구분하면 전체 구조를 이해하기 쉽습니다.

In [ ]:
# LlamaIndex 핵심 패키지 설치
# - llama-index-core: Document, Node, Index, QueryEngine 같은 기본 구성 요소를 제공합니다.
# - 이미 설치되어 있다면 실행하지 않아도 됩니다.
# !pip install llama-index-core

In [ ]:
# Ollama LLM 연동 패키지 설치
# - Ollama는 로컬에서 LLM을 실행하는 서버입니다.
# - 실행 전 터미널에서 `ollama serve`가 떠 있어야 합니다.
# - 사용할 모델도 미리 받아야 합니다: `ollama pull gemma2:2b`
# !pip install llama-index-llms-ollama

In [ ]:
# 임베딩 모델과 파일 리더 설치
# - nomic-embed-text 임베딩 모델도 미리 받아야 합니다: `ollama pull nomic-embed-text`
# - PDF 파일을 읽기 위해 llama-index-readers-file과 pypdf가 필요합니다.
# !pip install llama-index-embeddings-ollama llama-index-readers-file pypdf

<!-- 학습 보강 셀 -->

## 실행 전 준비 체크

이후 셀은 로컬 Ollama 서버와 모델이 준비되어 있어야 정상 실행됩니다.
터미널에서 `ollama serve`, `ollama pull gemma2:2b`, `ollama pull nomic-embed-text`를 먼저 확인하세요.
패키지 설치 셀은 한 번만 실행하면 되고, 커널을 새로 열었을 때는 import 셀부터 다시 실행하면 됩니다.

In [ ]:
# LlamaIndex에서 사용할 주요 객체를 불러옵니다.
# - Ollama: 질문에 대한 최종 답변을 생성하는 로컬 LLM 연결 객체
# - OllamaEmbedding: 문서를 벡터로 바꾸는 로컬 임베딩 모델 연결 객체
# - VectorStoreIndex: 문서 벡터를 저장하고 검색할 수 있게 만드는 인덱스
# - SimpleDirectoryReader: 디렉토리 안의 파일을 Document 객체로 읽어오는 로더
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

In [ ]:
# Ollama 모델 설정
# - model: 답변 생성에 사용할 LLM 이름입니다. 로컬 Ollama에 같은 이름의 모델이 있어야 합니다.
# - request_timeout: PDF 기반 질의는 오래 걸릴 수 있으므로 넉넉하게 둡니다.
# - temperature=0: 같은 질문에 최대한 일관된 답변이 나오도록 설정합니다.
llm = Ollama(
    model='gemma2:2b',
    request_timeout=120,
    temperature=0,
)

# 임베딩 모델 설정
# - 문서 검색 품질은 LLM보다 임베딩 모델에 크게 좌우됩니다.
embed_model = OllamaEmbedding(
    model_name='nomic-embed-text',
)

In [ ]:
# 문서 불러오기
# - 노트북 기준 상위 폴더의 NewData/pdf_sample1 안에 있는 PDF를 읽습니다.
# - SimpleDirectoryReader는 파일을 LlamaIndex Document 객체 목록으로 변환합니다.
documents = SimpleDirectoryReader(input_dir='../NewData/pdf_sample1').load_data()
print('읽어온 문서 수:', len(documents))

In [ ]:
# 첫 번째 문서의 메타데이터와 앞부분을 확인합니다.
# - 문서가 정상적으로 로드되었는지 확인한 뒤 인덱스를 만드는 순서가 안전합니다.
print(documents[0].metadata)
print(documents[0].text[:500])

<!-- 학습 보강 셀 -->

## Document 확인이 중요한 이유

RAG 오류는 모델보다 데이터 로드 단계에서 시작되는 경우가 많습니다.
본문이 비어 있거나 파일명이 잘못 들어오면, 인덱스는 만들어져도 검색 결과가 엉뚱해집니다.
따라서 인덱스를 만들기 전에 `문서 수`, `metadata`, `text 앞부분`을 확인하는 습관이 중요합니다.

In [ ]:
# 문서로부터 벡터 스토어 인덱스 생성
# - 각 Document가 여러 Node로 분할되고, Node별 임베딩 벡터가 생성됩니다.
# - 이 단계에서 Ollama 임베딩 서버가 호출됩니다.
index = VectorStoreIndex.from_documents(
    documents,
    embed_model=embed_model,
    show_progress=True,
)

<!-- 학습 보강 셀 -->

## 인덱스 생성 단계에서 일어나는 일

`VectorStoreIndex.from_documents()`는 단순 저장 함수가 아닙니다.
내부적으로 문서를 검색 가능한 작은 단위로 나누고, 각 조각을 임베딩 벡터로 변환한 뒤, 질문과 비교할 수 있는 검색 구조를 만듭니다.
이 단계가 느리다면 대부분 임베딩 모델 호출 시간이 원인입니다.

In [ ]:
index

In [ ]:
# 쿼리 엔진 생성
# - 검색된 문서 조각을 LLM에 전달해 답변을 생성하는 인터페이스입니다.
query_engine = index.as_query_engine(llm=llm)

In [ ]:
# 응답 생성
# - 질문과 관련된 Node를 검색한 뒤, 검색 결과를 바탕으로 LLM이 답변합니다.
query = '미국의 인공지능 정책과 주요 변화에 대해 알려줘'
response = query_engine.query(query)
print(response)

<!-- 학습 보강 셀 -->

## 답변만 보지 말고 근거를 같이 확인하기

RAG에서는 최종 답변보다 `어떤 문서 조각을 근거로 답했는지`가 더 중요할 때가 많습니다.
다음 셀의 metadata와 source_nodes를 보면 답변이 실제 문서에 기반했는지, 아니면 관련 없는 조각을 보고 생성됐는지 판단할 수 있습니다.

In [ ]:
# 응답 메타데이터 확인
# - 어떤 파일/페이지/노드가 답변 근거로 사용되었는지 추적할 때 활용합니다.
response.metadata

In [ ]:
# 검색 근거와 점수 확인
# - score는 검색된 source_node와 질문 사이의 관련도입니다.
# - 값의 해석은 사용하는 벡터 스토어와 검색 방식에 따라 달라질 수 있습니다.
for i, node in enumerate(response.source_nodes, start=1):
    print(f'근거 {i} score: {node.score}')
    print(node.node.text[:300])
    print('-' * 80)